# Progetto DIA - A.A 2025/26

Autori: Justin Carideo (justin.carideo@studio.unibo.it), Laura Bertozzi ()

# Setup

In [10]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import sklearn

Importiamo i dataset presi dal sito https://www.kaggle.com. Questi due dataset fanno riferimento a dati in ambito agricolo, in particolare:
1) **Crop Dataset** -> dataset incentrato su aspetti ambientali per individuare il tipo di coltura dato il tipo di terreno e nutrienti presenti (https://www.kaggle.com/datasets/snmahsa/soil-nutrients)
2) **Fertilizer DataSet** -> dataset incentrato sullo stesso campo ma che vuole individuare il tipo di fertilizzante dato il tipo di terreno e la coltura presente. (https://www.kaggle.com/datasets/nishchalchandel/fertilizer-recommendation)

In [11]:
import kaggle
import os
import os.path

CROP_DATASET_ID = "varshitanalluri/crop-recommendation-dataset"
FERTILIZER_DATASET_ID = "nishchalchandel/fertilizer-recommendation"

BASE_DOWNLOAD_DIR = "data"

CROP_DATASET_DIR = os.path.join(BASE_DOWNLOAD_DIR, "dataset_1")
FERTILIZER_DATASET_DIR = os.path.join(BASE_DOWNLOAD_DIR, "dataset_2")

os.makedirs(BASE_DOWNLOAD_DIR, exist_ok=True) # crea la directory se non esiste


def download_kaggle_dataset(dataset_id, dataset_path):
    os.makedirs(dataset_path, exist_ok=True)
    if not os.listdir(dataset_path):
        print(f"Download {dataset_id} in {dataset_path}")
        kaggle.api.dataset_download_files(
            dataset_id,
            path=dataset_path,
            unzip=True
        )
    else:
        print(f"Dataset already in specified path")

download_kaggle_dataset(CROP_DATASET_ID, CROP_DATASET_DIR)
download_kaggle_dataset(FERTILIZER_DATASET_ID, FERTILIZER_DATASET_DIR)

CROP_DATASET_PATH = os.path.join(CROP_DATASET_DIR, "Crop_recommendation.csv")
FERTILIZER_DATASET_PATH = os.path.join(FERTILIZER_DATASET_DIR, "fertilizer_recommendation_dataset.csv")

Dataset already in specified path
Dataset already in specified path


Creiamo i dataframe di entrambi i dataset:

In [12]:
crop_df = pd.read_csv(CROP_DATASET_PATH)
fertilizer_df = pd.read_csv(FERTILIZER_DATASET_PATH)

Ora che abbiamo ottenuto i dataset dobbiamo decidere che modello attuare su questi dataset e visto che non hanno indici univoci per riga possiamo attuare una **recommendation** per la predizione di colture e fertilizzanti. In particolare potremmo ragionare con una sorta di *pipeline* dove i dati di entrambi vengono utilizzati per ottenere una classificazione della coltura desiderata date le informazioni relative al terreno e utilizzare i dati ottenuti per calcolare anche il tipo di fertilizzante da attuare.

Questa è una analisi a priori, una volta svolta l'*analisi esplorativa* si potrà procedere con la costruzione del modello.

# Analisi Esplorativa

In [13]:
crop_df

,Nitrogen,Phosphorus,Potassium,Temperature,Humidity,pH_Value,Rainfall,Crop
0,90,42,43,20.879744,82.002744,6.502985,202.935536,Rice
1,85,58,41,21.770462,80.319644,7.038096,226.655537,Rice
2,60,55,44,23.004459,82.320763,7.840207,263.964248,Rice
3,74,35,40,26.491096,80.158363,6.980401,242.864034,Rice
4,78,42,42,20.130175,81.604873,7.628473,262.717340,Rice
...,...,...,...,...,...,...,...,...
2195,107,34,32,26.774637,66.413269,6.780064,177.774507,Coffee
2196,99,15,27,27.417112,56.636362,6.086922,127.924610,Coffee
2197,118,33,30,24.131797,67.225123,6.362608,173.322839,Coffee
2198,117,32,34,26.272418,52.127394,6.758793,127.175293,Coffee


In [14]:
fertilizer_df

,Temperature,Moisture,Rainfall,PH,Nitrogen,Phosphorous,Potassium,Carbon,Soil,Crop,Fertilizer,Remark
0,50.179845,0.725893,205.600816,6.227358,66.701872,76.963560,96.429065,0.496300,Loamy Soil,rice,Compost,Enhances organic matter and improves soil stru...
1,21.633318,0.721958,306.081601,7.173131,71.583316,163.057636,148.128347,1.234242,Loamy Soil,rice,Balanced NPK Fertilizer,"Provides a balanced mix of nitrogen, phosphoru..."
2,23.060964,0.685751,259.336414,7.380793,75.709830,62.091508,80.308971,1.795650,Peaty Soil,rice,Water Retaining Fertilizer,Improves water retention in dry soils. Prefer ...
3,26.241975,0.755095,212.703513,6.883367,78.033687,151.012521,153.005712,1.517556,Loamy Soil,rice,Balanced NPK Fertilizer,"Provides a balanced mix of nitrogen, phosphoru..."
4,21.490157,0.730672,268.786767,7.578760,71.765123,66.257371,97.000886,1.782985,Peaty Soil,rice,Organic Fertilizer,"Enhances fertility naturally, ideal for peaty ..."
...,...,...,...,...,...,...,...,...,...,...,...,...
3095,23.486430,0.531191,46.412228,6.733584,56.534283,146.111078,81.389366,1.602913,Neutral Soil,watermelon,Water Retaining Fertilizer,Improves water retention in dry soils. Prefer ...
3096,24.289508,0.736699,63.068103,6.372709,56.358005,49.003277,46.695889,1.473656,Peaty Soil,watermelon,DAP,"Rich in phosphorus, essential for root develop..."
3097,23.945488,0.520513,41.344590,7.051515,55.738905,148.567285,90.057021,1.455045,Neutral Soil,watermelon,Water Retaining Fertilizer,Improves water retention in dry soils. Prefer ...
3098,26.368604,0.547436,33.106012,6.615922,57.711705,96.662953,59.531473,0.614487,Acidic Soil,watermelon,Compost,Enhances organic matter and improves soil stru...


In [5]:
crop_df.describe()

,Temparature,Humidity,Moisture,Nitrogen,Potassium,Phosphorous
count,8000.000000,8000.000000,8000.000000,8000.000000,8000.000000,8000.000000
mean,30.338895,59.210731,43.580862,18.429125,3.916375,18.512500
std,4.478262,8.177366,12.596156,11.852406,5.494807,13.244113
min,20.000000,40.020000,20.000000,0.000000,0.000000,0.000000
25%,27.050000,53.277500,33.967500,9.000000,0.000000,8.000000
50%,30.240000,59.110000,42.250000,14.000000,1.000000,18.000000
75%,33.460000,65.082500,52.950000,26.000000,5.000000,30.000000
max,40.000000,80.000000,70.000000,46.000000,23.000000,46.000000


La parte più importante a quanto pare sono le colture perché non hanno una corrispondenza biunivoca, di conseguenza dovremmo capire come mappare i dati tra i due dataset.

In [20]:
crops_fertilizer = fertilizer_df["Crop"].unique()
print(crops_fertilizer)

<StringArray>
[        'rice',        'wheat',    'Mung Bean',          'Tea',
       'millet',        'maize',       'Lentil',         'Jute',
       'Coffee',       'Cotton',   'Ground Nut',         'Peas',
       'Rubber',    'Sugarcane',      'Tobacco', 'Kidney Beans',
   'Moth Beans',      'Coconut',   'Black gram', 'Adzuki Beans',
  'Pigeon Peas',     'Chickpea',       'banana',       'grapes',
        'apple',        'mango',    'muskmelon',       'orange',
       'papaya',  'pomegranate',   'watermelon']
Length: 31, dtype: str


In [21]:
crops_crop = crop_df["Crop"].unique()
print(crops_crop)

<StringArray>
[       'Rice',       'Maize',    'ChickPea', 'KidneyBeans',  'PigeonPeas',
   'MothBeans',    'MungBean',   'Blackgram',      'Lentil', 'Pomegranate',
      'Banana',       'Mango',      'Grapes',  'Watermelon',   'Muskmelon',
       'Apple',      'Orange',      'Papaya',     'Coconut',      'Cotton',
        'Jute',      'Coffee']
Length: 22, dtype: str


Notiamo che i due dataset non hanno nessuna corrispondenza a livello di colture coltivate ma notiamo che le colture di `crop_df` rientrano perfettamente nelle colture di `fertilizer_df`. Di conseguenza bisogna mapparli in maniera tale da poter ragionare con gli stessi dati.

In [22]:
def clean_crop_name(name):
    return str(name).lower().replace(" ", "")

In [28]:
crop_df["Crop"].map(clean_crop_name).unique()

<StringArray>
[       'rice',       'maize',    'chickpea', 'kidneybeans',  'pigeonpeas',
   'mothbeans',    'mungbean',   'blackgram',      'lentil', 'pomegranate',
      'banana',       'mango',      'grapes',  'watermelon',   'muskmelon',
       'apple',      'orange',      'papaya',     'coconut',      'cotton',
        'jute',      'coffee']
Length: 22, dtype: str

In [29]:
fertilizer_df["Crop"].map(clean_crop_name).unique()

<StringArray>
[       'rice',       'wheat',    'mungbean',         'tea',      'millet',
       'maize',      'lentil',        'jute',      'coffee',      'cotton',
   'groundnut',        'peas',      'rubber',   'sugarcane',     'tobacco',
 'kidneybeans',   'mothbeans',     'coconut',   'blackgram', 'adzukibeans',
  'pigeonpeas',    'chickpea',      'banana',      'grapes',       'apple',
       'mango',   'muskmelon',      'orange',      'papaya', 'pomegranate',
  'watermelon']
Length: 31, dtype: str